In [15]:
import os
import numpy as np
import pandas as pd
from keras_preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.utils import class_weight
from tensorflow.keras import regularizers
from tensorflow.keras.applications import VGG16

# Define paths
train_path = r"C:\Users\somchay\projuctworkshop\Train"
test_path = r"C:\Users\somchay\projuctworkshop\Test"

# Image Size and batch size
Image_Size = (200, 200)
batch_size = 32

# Data augmentation for training data
train_datagen = ImageDataGenerator(
    rotation_range=15,
    rescale=1.0 / 255,
    shear_range=0.1,
    zoom_range=0.2,
    horizontal_flip=True,
    width_shift_range=0.1,
    height_shift_range=0.1
)

train_generator = train_datagen.flow_from_directory(
    train_path,
    target_size=Image_Size,
    class_mode='binary',
    batch_size=batch_size
)

# Data augmentation for validation data
validation_datagen = ImageDataGenerator(rescale=1.0 / 255)
validation_generator = validation_datagen.flow_from_directory(
    test_path,
    target_size=Image_Size,
    class_mode='binary',
    batch_size=batch_size
)

# ตรวจสอบค่าคลาสที่ได้จาก train_generator
print("Unique classes:", np.unique(train_generator.classes))
print("Data type of classes:", train_generator.classes.dtype)

# แปลงค่า classes ให้เป็น int และคำนวณ class weights
classes = np.unique(train_generator.classes).astype(int)
class_weights = class_weight.compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=train_generator.classes.astype(int)
)
class_weights = dict(zip(classes, class_weights))

print("Class Weights:", class_weights)


base_model = VGG16(weights='imagenet', include_top=False, input_shape=(Image_Size[0], Image_Size[1], 3))

# Freeze the base model
for layer in base_model.layers:
    layer.trainable = False

# Define the model with Transfer Learning
model = Sequential([
    base_model,
    Flatten(),
    Dense(128, activation='relu', kernel_regularizer=regularizers.l2(0.01)),
    Dropout(0.5),
    Dense(1, activation='sigmoid')  # ใช้ 'sigmoid' สำหรับการจำแนกประเภท 2 คลาส
])

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Early stopping and learning rate reduction
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=1e-6)

# Train the model with class weights
history = model.fit(
    train_generator,
    epochs=50,
    validation_data=validation_generator,
    class_weight=class_weights,
    callbacks=[early_stopping, reduce_lr]
)

# Save the model in .keras format
model.save(r'C:\Users\somchay\projuctworkshop\AIproject.keras')

# แสดงความแม่นยำ
train_accuracy = history.history['accuracy'][-1]
val_accuracy = history.history['val_accuracy'][-1]

print(f"Training accuracy: {train_accuracy:.2f}")
print(f"Validation accuracy: {val_accuracy:.2f}")


Found 804 images belonging to 2 classes.
Found 804 images belonging to 2 classes.
Unique classes: [0 1]
Data type of classes: int32
Class Weights: {0: 1.0, 1: 1.0}


D:\Anaconda3\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 124s 5s/step - accuracy: 0.6051 - loss: 2.8267 - val_accuracy: 0.9988 - val_loss: 0.7066 - learning_rate: 0.0010
Epoch 2/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 128s 5s/step - accuracy: 0.9646 - loss: 0.7296 - val_accuracy: 1.0000 - val_loss: 0.4571 - learning_rate: 0.0010
Epoch 3/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 136s 5s/step - accuracy: 0.9818 - loss: 0.4890 - val_accuracy: 0.9988 - val_loss: 0.3270 - learning_rate: 0.0010
Epoch 4/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 138s 5s/step - accuracy: 0.9912 - loss: 0.3329 - val_accuracy: 0.9988 - val_loss: 0.2434 - learning_rate: 0.0010
Epoch 5/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 140s 5s/step - accuracy: 0.9938 - loss: 0.2643 - val_accuracy: 1.0000 - val_loss: 0.1930 - learning_rate: 0.0010
Epoch 6/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 142s 6s/step - accuracy: 0.9966 - loss: 0.2133 - val_accuracy: 1.0000 - val_loss: 0.1578 - learning_rate: 0.0010
Epoch 7/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 144s 6s/step - accuracy: 0.9948 - loss: 0.1839 - val_acc

In [ ]:
import cv2
import numpy as np
import os
from tensorflow.keras.models import load_model

# โหลดโมเดล
model = load_model(r'C:\Users\somchay\projuctworkshop\AIproject.keras')  # ใช้ raw string

# ค้นหาภาพทั้งหมดในโฟลเดอร์ Test
test_folder = r"C:\Users\somchay\projuctworkshop\Test"
test_images = []
for root, dirs, files in os.walk(test_folder):
    for file in files:
        if file.lower().endswith(('.png', '.jpg', '.jpeg')):
            test_images.append(os.path.join(root, file))

# กำหนด Threshold
THRESHOLD = 0.7  # เปลี่ยนเป็น 0.7 สำหรับการเปรียบเทียบกับค่าความมั่นใจ

# ทำนายและแสดงผล
for img_path in test_images:
    image = cv2.imread(img_path)
    if image is None:
        print(f"Error: ไม่พบไฟล์ {img_path}")
        continue
    
    # แปลงสีและประมวลผลภาพ
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    resized_img = cv2.resize(image_rgb, (200, 200))  # ปรับขนาดให้ตรงกับโมเดล
    normalized_img = resized_img.astype("float32") / 255.0
    input_img = np.expand_dims(normalized_img, axis=0)
    
    # ทำนายผล
    prediction = model.predict(input_img, verbose=0)
    
    # สำหรับ binary classification
    normal_confidence = prediction[0][0] * 100  
    funny_confidence = (1 - prediction[0][0]) * 100  

    # กำหนด label
    if funny_confidence > THRESHOLD * 100:  
        label = f"Funny ({funny_confidence:.1f}%)"  
    elif normal_confidence > THRESHOLD * 100:
        label = f"Normal ({normal_confidence:.1f}%)"
    else:
        label = "Uncertain"

    # แสดงผล
    print(f"Image: {img_path}")
    print(f"Actual Class: {os.path.basename(os.path.dirname(img_path))}")
    print(f"Prediction: {label}")
    print("----------------------------------")


Image: C:\Users\somchay\projuctworkshop\Test\Funny Face\IMG_1649.jpg
Actual Class: Funny Face
Prediction: Funny (90.4%)
----------------------------------
Image: C:\Users\somchay\projuctworkshop\Test\Funny Face\IMG_1650.jpg
Actual Class: Funny Face
Prediction: Funny (99.2%)
----------------------------------
Image: C:\Users\somchay\projuctworkshop\Test\Funny Face\IMG_1651.jpg
Actual Class: Funny Face
Prediction: Funny (98.8%)
----------------------------------
Image: C:\Users\somchay\projuctworkshop\Test\Funny Face\IMG_1652.jpg
Actual Class: Funny Face
Prediction: Funny (99.1%)
----------------------------------
Image: C:\Users\somchay\projuctworkshop\Test\Funny Face\IMG_1653.jpg
Actual Class: Funny Face
Prediction: Funny (98.8%)
----------------------------------
Image: C:\Users\somchay\projuctworkshop\Test\Funny Face\IMG_1654.jpg
Actual Class: Funny Face
Prediction: Funny (99.8%)
----------------------------------
Image: C:\Users\somchay\projuctworkshop\Test\Funny Face\IMG_1655.jpg
A

In [3]:
import cv2
import numpy as np
from tensorflow.keras.models import load_model

# โหลดโมเดล
model = load_model(r'C:\Users\somchay\projuctworkshop\AIproject.keras')

# กำหนด Threshold
THRESHOLD = 0.7

# เปิดกล้อง
cap = cv2.VideoCapture(0)

# โหลด Haar Cascade สำหรับการตรวจจับใบหน้า
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

while True:
    # จับภาพจากกล้อง
    ret, frame = cap.read()
    if not ret:
        print("Error: ไม่สามารถจับภาพจากกล้องได้")
        break

    # แปลงสีและประมวลผลภาพ
    gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)  
    faces = face_cascade.detectMultiScale(gray_frame, scaleFactor=1.1, minNeighbors=5)

    for (x, y, w, h) in faces:
        # ตัดภาพใบหน้าเพื่อทำการพยากรณ์
        face_roi = frame[y:y + h, x:x + w]
        resized_img = cv2.resize(face_roi, (200, 200))  
        normalized_img = resized_img.astype("float32") / 255.0
        input_img = np.expand_dims(normalized_img, axis=0)

        # ทำนายผล
        prediction = model.predict(input_img, verbose=0)

        # คำนวณค่าความมั่นใจ
        funny_confidence = (1 - prediction[0][0]) * 100  
        normal_confidence = prediction[0][0] * 100  

        # กำหนด label และสี
        if funny_confidence > THRESHOLD * 100:
            label = f"Funny ({funny_confidence:.1f}%)"
            color = (0, 255, 0)  # หน้า funny สีเขียวอี๊
        elif normal_confidence > THRESHOLD * 100:
            label = f"Normal ({normal_confidence:.1f}%)"
            color = (0, 255, 255)  # หน้าnormal  สีเหลืองอึงงง
        else:
            label = "Uncertain"
            color = (0, 0, 255)  # หน้าที่ uncertainสีแดงจี๊ด จ๊าดดด

        # วาดกรอบสี่เหลี่ยมรอบใบหน้าด้วยสีที่กำหนด
        cv2.rectangle(frame, (x, y), (x + w, y + h), color, 2)

        # แสดงผลบนภาพ
        cv2.putText(frame, label, (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)

    # แสดงผลภาพ
    cv2.imshow('Webcam', frame)

    # ออกจากลูปเมื่อกด 'q'
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# ปิดกล้องและหน้าต่าง
cap.release()
cv2.destroyAllWindows()
